In [ ]:
import argparse
import os
import sys

import requests

#func to print to stderr
def printError(msg, code=1):
    print(f"ERROR: {msg}", file=sys.stderr)
    sys.exit(code)
  
#checks for existence of desired file and then uploads it via post request
def upload(args):
    url = args.base_url.rstrip("/") + "/upload"
    if not os.path.exists(args.file): #check for file
        printError(f"file not found: {args.file}")

    files = {"file": open(args.file, "rb")}
    data = {}
    if args.srid is not None:
        data["srid"] = str(args.srid) #checks in case of custom srid

    r = requests.post(url, files=files, data=data, timeout=120) #post request
    if r.status_code >= 400:
        printError(f"upload failed ({r.status_code}): {r.text}")

    print(r.text)


def CMDquery(args):
    url = args.base_url.rstrip("/") + "/query" #build the query url from the base url

    if len(args.bbox) != 4: 
        printError("bbox must have 4 numbers")
    xmin, ymin, xmax, ymax = map(float, args.bbox) 
    if xmin > xmax or ymin > ymax: #check ordering is valid
        printError("invalid bbox")

    if len(args.z) != 2: 
        printError("z must have 2 numbers")
    zmin, zmax = map(float, args.z) #convert to floats
    if zmin > zmax: #check z ordering is valid
        printError("invalid z ordering")

    params = { #package all 6 coords as query params for the GET request
        "xmin": xmin,
        "ymin": ymin,
        "xmax": xmax,
        "ymax": ymax,
        "zmin": zmin,
        "zmax": zmax,}

    r = requests.get(url, params=params, timeout=60) 
    if r.status_code >= 400: #check for error 
        printError(f"query failure ({r.status_code}): {r.text}")

    payload = r.json()
    matches = payload.get("matches", []) #get the list of matching files
    print(f"matches: {len(matches)}")
    for m in matches:
        fp = m.get("file_path") if isinstance(m, dict) else None #get file path if row is a dict
        mid = m.get("id") if isinstance(m, dict) else None #get file id if row is a dict
        if fp is not None:
            print(f"- id={mid} path={fp}") #print id and path
        else:
            print(f"- {m}") #fallback if row format is unexpected


def BuildParser():
    p = argparse.ArgumentParser("COPCproj",description="Command line tool for COPC ingest + bbox querying using Flask API",)
    p.add_argument("BaseUrl-",default="http://localhost:5000",help="API base URL (default: http://localhost:5000)",)

    sub = p.add_subparsers(dest="cmd", required=True)

    up = sub.add_parser("upload", help="Upload/ingest a COPC/LAZ file")
    up.add_argument("file", help="Path to .laz/.copc.laz")
    up.add_argument("--srid", type=int, default=None, help="Optional SRID if your API expects it")
    up.set_defaults(func=upload)

    q = sub.add_parser("query", help="Query intersecting tiles by bbox + z-range")
    q.add_argument(
        "--bbox",
        nargs=4,
        required=True,
        metavar=("XMIN", "YMIN", "XMAX", "YMAX"),
        help="2D bbox (must match your DB footprint CRS, e.g. TM65 EPSG:29902)",
    )
    q.add_argument(
        "--z",
        nargs=2,
        required=True,
        metavar=("ZMIN", "ZMAX"),
        help="Z range",
    )
    q.add_argument("--raw", action="store_true", help="Print raw JSON response")
    q.set_defaults(func=CMDquery)

    return p


def main():
    parser = BuildParser()
    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()